# 06 — CNN sobre mapas de puntos de la mano

Este notebook es autocontenido y está preparado para Google Colab. MediaPipe localiza la mano y sus 21 puntos; los puntos se dibujan como un esqueleto RGB normalizado sobre fondo negro. Luego se comparan **MobileNetV2** y **EfficientNetB0** con transferencia de aprendizaje.

El experimento evalúa imágenes estáticas. La augmentación cambia únicamente brillo y contraste; **no rota ni voltea** los mapas. Las fallas del detector se contabilizan en la evaluación completa del sistema.

## Antes de ejecutarlo

1. En Colab, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.
2. Guarda o copia el dataset a Google Drive. Ajusta `DRIVE_ROOT` abajo para que `RAW_DIR` señale a la carpeta que contiene una subcarpeta por clase (`A/`, `B/`, ..., `nothing/`, `space/`, `del/`).
3. Ejecuta primero el piloto de cinco imágenes por clase, revisa los mapas y luego procesa el lote completo. La extracción de todo el dataset puede tardar; los mapas se guardan en Drive y el script puede continuar una extracción interrumpida.
4. Ejecuta las celdas de entrenamiento de MobileNetV2 y EfficientNetB0 en orden. Los pesos ImageNet se descargan la primera vez.

Las imágenes y resultados generados quedan en `data/processed/landmark_map_experiment/` dentro de Drive. No hay que subirlos a GitHub.

In [ ]:
!apt-get update -qq && apt-get install -y -qq libgles2 libegl1
%pip install -q mediapipe==0.10.35 scikit-learn

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import random
import shutil
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import mediapipe as mp
from PIL import Image, ImageDraw, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Cambia esta carpeta si guardaste el repositorio/dataset en otra ruta de Drive.
DRIVE_ROOT = Path("/content/drive/MyDrive/proyecto-ia2")
RAW_DIR = DRIVE_ROOT / "data/raw/asl_alphabet_train/asl_alphabet_train"
OWN_PHOTOS_DIR = DRIVE_ROOT / "data/mis_fotos"  # Opcional: carpetas por clase para prueba externa.
OUTPUT_DIR = DRIVE_ROOT / "data/processed/landmark_map_experiment"
MAP_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 8

if not tf.config.list_physical_devices("GPU"):
    print("No se detectó GPU de TensorFlow. El notebook puede ejecutarse en CPU, pero tardará más.")
else:
    print("GPU de TensorFlow:", tf.config.list_physical_devices("GPU"))

print("RAW_DIR:", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Verificar el dataset

La carpeta indicada debe contener las clases directamente. Si el conteo queda muy por debajo de unas 87.000 imágenes, revisa si el dataset está incompleto o si `RAW_DIR` apunta a una subcarpeta parcial antes de generar mapas.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f"No encuentro {RAW_DIR}. Ajusta DRIVE_ROOT/RAW_DIR y vuelve a ejecutar esta celda."
    )

raw_class_dirs = sorted(p for p in RAW_DIR.iterdir() if p.is_dir())
raw_counts = {
    p.name: sum(1 for f in p.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)
    for p in raw_class_dirs
}
class_names = [name for name, count in raw_counts.items() if count > 0]
print(f"Clases: {len(class_names)} | imágenes encontradas: {sum(raw_counts.values()):,}")
display(pd.DataFrame({"clase": list(raw_counts), "imágenes": list(raw_counts.values())}))
if len(class_names) != 29:
    warnings.warn(f"Se esperaban 29 clases de ASL; se encontraron {len(class_names)}. Verifica la carpeta del dataset.")
if sum(raw_counts.values()) < 80000:
    warnings.warn("Hay menos imágenes de las esperadas para el dataset completo; confirma que la descarga/extracción terminó.")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ASSET = OUTPUT_DIR / "hand_landmarker.task"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/latest/hand_landmarker.task"
)
if not MODEL_ASSET.exists():
    print("Descargando el modelo de MediaPipe...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_ASSET)
print("Modelo MediaPipe:", MODEL_ASSET, f"({MODEL_ASSET.stat().st_size / 1e6:.1f} MB)")

## 2. Extraer puntos y dibujar mapas

Cada mano se centra en la muñeca y se escala por el tamaño de la palma; esto reduce las diferencias de ubicación y tamaño. El dibujo conserva la orientación de la mano. No conserva coordenadas `z`: esta versión compara mapas 2D de los 21 puntos.

In [ ]:
FINGER_COLORS = [
    (255, 90, 90),    # pulgar
    (255, 210, 40),   # índice
    (80, 235, 100),   # medio
    (50, 190, 255),   # anular
    (220, 90, 255),   # meñique
]
FINGER_EDGES = [
    [(0, 1), (1, 2), (2, 3), (3, 4)],
    [(0, 5), (5, 6), (6, 7), (7, 8)],
    [(0, 9), (9, 10), (10, 11), (11, 12)],
    [(0, 13), (13, 14), (14, 15), (15, 16)],
    [(0, 17), (17, 18), (18, 19), (19, 20)],
]
PALM_EDGES = [(5, 9), (9, 13), (13, 17)]
POINT_COLORS = {
    **{i: FINGER_COLORS[0] for i in range(1, 5)},
    **{i: FINGER_COLORS[1] for i in range(5, 9)},
    **{i: FINGER_COLORS[2] for i in range(9, 13)},
    **{i: FINGER_COLORS[3] for i in range(13, 17)},
    **{i: FINGER_COLORS[4] for i in range(17, 21)},
    0: (255, 255, 255),
}


def landmarks_to_map(hand_landmarks, size=MAP_SIZE):
    """Rasteriza 21 puntos y conexiones en una imagen RGB; None si la geometría no es válida."""
    xy = np.asarray([(p.x, p.y) for p in hand_landmarks], dtype=np.float32)
    if xy.shape != (21, 2) or not np.isfinite(xy).all():
        return None
    xy -= xy[0]  # origen en la muñeca
    palm_scale = float(np.linalg.norm(xy[9]))  # muñeca a base del dedo medio
    if palm_scale < 1e-6:
        return None
    xy /= palm_scale
    extent = float(np.max(np.abs(xy)))
    if extent < 1e-6:
        return None
    xy = xy / extent * 0.40 * (size - 1) + (size - 1) / 2
    points = [(int(round(x)), int(round(y))) for x, y in xy]

    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    draw = ImageDraw.Draw(canvas)
    line_width = max(2, size // 70)
    radius = max(3, size // 48)
    for edges, color in zip(FINGER_EDGES, FINGER_COLORS):
        for a, b in edges:
            draw.line((points[a], points[b]), fill=color, width=line_width)
    for a, b in PALM_EDGES:
        draw.line((points[a], points[b]), fill=(235, 235, 235), width=line_width)
    for i, (x, y) in enumerate(points):
        color = POINT_COLORS[i]
        draw.ellipse((x-radius, y-radius, x+radius, y+radius), fill=color, outline=(255, 255, 255))
    return canvas


def canonical_label(folder_name, names):
    """Alinea nombres comunes de las carpetas propias con las clases de Kaggle."""
    key = folder_name.strip().lower()
    if key == "delete":
        key = "del"
    lookup = {name.lower(): name for name in names}
    return lookup.get(key)


def collect_records(root, names=None, max_per_class=None, seed=SEED):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"No existe la carpeta: {root}")
    records = []
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        label = canonical_label(class_dir.name, names) if names is not None else class_dir.name
        if label is None:
            continue
        files = sorted(p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        if max_per_class is not None and len(files) > max_per_class:
            rng = np.random.default_rng(seed + sum(map(ord, label)))
            chosen = np.sort(rng.choice(len(files), size=max_per_class, replace=False))
            files = [files[i] for i in chosen]
        for path in files:
            relative = path.relative_to(root).as_posix()
            suffix = path.suffix.lower().lstrip(".")
            map_relative = f"{label}/{path.stem}_{suffix}_map.png"
            records.append({"source": path, "path": relative, "label": label, "map_path": map_relative})
    if not records:
        raise FileNotFoundError(f"No se hallaron imágenes por clase en {root}")
    return records


class HandMapExtractor:
    def __init__(self, model_path):
        options = mp.tasks.vision.HandLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),
            running_mode=mp.tasks.vision.RunningMode.IMAGE,
            num_hands=1,
            min_hand_detection_confidence=0.5,
            min_hand_presence_confidence=0.5,
        )
        self.detector = mp.tasks.vision.HandLandmarker.create_from_options(options)

    def extract(self, image_path):
        with Image.open(image_path) as source:
            rgb = np.asarray(ImageOps.exif_transpose(source).convert("RGB"))
        result = self.detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        if not result.hand_landmarks:
            return Image.new("RGB", (MAP_SIZE, MAP_SIZE), (0, 0, 0)), False
        mapped = landmarks_to_map(result.hand_landmarks[0])
        if mapped is None:
            return Image.new("RGB", (MAP_SIZE, MAP_SIZE), (0, 0, 0)), False
        return mapped, True

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

    def close(self):
        self.detector.close()


def prepare_maps(input_root, maps_root, manifest_path, model_path, names=None, max_per_class=None, progress_every=500):
    """Genera PNG y añade filas a un manifiesto; reutiliza mapas ya procesados al reanudar."""
    records = collect_records(input_root, names, max_per_class)
    maps_root, manifest_path = Path(maps_root), Path(manifest_path)
    maps_root.mkdir(parents=True, exist_ok=True)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    existing = {}
    if manifest_path.exists():
        old = pd.read_csv(manifest_path).drop_duplicates("path", keep="last")
        existing = {str(row.path): row._asdict() for row in old.itertuples(index=False)}

    write_header = not manifest_path.exists() or manifest_path.stat().st_size == 0
    processed, detected_count, failures = 0, 0, 0
    with HandMapExtractor(model_path) as extractor, manifest_path.open("a", newline="", encoding="utf-8") as stream:
        import csv
        writer = csv.DictWriter(stream, fieldnames=["path", "label", "map_path", "detected", "error"])
        if write_header:
            writer.writeheader()
        for i, row in enumerate(records, 1):
            destination = maps_root / row["map_path"]
            old_row = existing.get(row["path"])
            if old_row and destination.is_file() and str(old_row.get("label")) == row["label"]:
                detected = bool(int(old_row["detected"]))
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                error_text = ""
                try:
                    mapped, detected = extractor.extract(row["source"])
                except Exception as exc:
                    mapped, detected = Image.new("RGB", (MAP_SIZE, MAP_SIZE), (0, 0, 0)), False
                    error_text = f"{type(exc).__name__}: {exc}"
                    failures += 1
                mapped.save(destination, format="PNG")
                writer.writerow({
                    "path": row["path"], "label": row["label"], "map_path": row["map_path"],
                    "detected": int(detected), "error": error_text,
                })
                if i % 100 == 0:
                    stream.flush()
            processed += 1
            detected_count += int(detected)
            if i % progress_every == 0 or i == len(records):
                stream.flush()
                print(f"Procesadas {i:,}/{len(records):,} | detector encontró mano: {detected_count:,}")
    print(f"Mapas guardados/reutilizados: {processed:,} | errores de lectura: {failures}")
    return pd.read_csv(manifest_path).drop_duplicates("path", keep="last")

## 3. Piloto visual

Procesa cinco imágenes por clase primero. El piloto usa una carpeta y un manifiesto aparte, así no se mezcla con el lote completo.

In [ ]:
PILOT_MAPS = OUTPUT_DIR / "maps_pilot"
PILOT_MANIFEST = OUTPUT_DIR / "maps_pilot_manifest.csv"
pilot_df = prepare_maps(
    RAW_DIR, PILOT_MAPS, PILOT_MANIFEST, MODEL_ASSET,
    names=class_names, max_per_class=5,
)
print("Cobertura del detector en piloto:")
display(pilot_df.groupby("label").detected.agg(["sum", "count", "mean"]).rename(columns={"mean": "tasa_detectada"}))

sample_rows = pilot_df.sample(min(12, len(pilot_df)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, row in zip(axes.flat, sample_rows.itertuples(index=False)):
    ax.imshow(Image.open(PILOT_MAPS / row.map_path))
    ax.set_title(f"{row.label} | mano={bool(row.detected)}", fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(sample_rows):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Procesar el dataset completo

Si el piloto muestra esqueleto completo y la tasa de detección es razonable, procesa todas las imágenes. Si Colab se desconecta, vuelve a ejecutar esta celda: los mapas presentes y registrados en el manifiesto se reutilizan.

In [ ]:
FULL_MAPS = OUTPUT_DIR / "maps_full"
FULL_MANIFEST = OUTPUT_DIR / "maps_full_manifest.csv"
full_df = prepare_maps(
    RAW_DIR, FULL_MAPS, FULL_MANIFEST, MODEL_ASSET,
    names=class_names, max_per_class=None, progress_every=500,
)
coverage = full_df.groupby("label").detected.agg(["sum", "count", "mean"])
coverage = coverage.rename(columns={"sum": "manos_detectadas", "count": "imágenes", "mean": "tasa_detectada"})
display(coverage)
print(f"Total de mapas: {len(full_df):,} | tasa de detección total: {full_df.detected.mean():.2%}")

## 5. Partición estratificada y `tf.data`

Se divide por imagen una sola vez (70 % entrenamiento, 15 % validación, 15 % prueba) y se guarda `splits.csv`. Las clases con imagen pero sin mano detectada se excluyen del ajuste/validación del clasificador; se mantienen en la prueba y se predicen como `nothing` para medir el flujo completo. La clase `nothing` permanece en todas las particiones.

In [ ]:
SPLIT_FILE = OUTPUT_DIR / "splits.csv"
full_df = full_df.drop_duplicates("path", keep="last").reset_index(drop=True)
if SPLIT_FILE.exists():
    splits = pd.read_csv(SPLIT_FILE)
    # Si el manifiesto cambió de tamaño, rehacer las divisiones para cubrir todas las rutas actuales.
    if set(splits.path.astype(str)) != set(full_df.path.astype(str)):
        print("El manifiesto cambió; se crearán nuevas divisiones estratificadas.")
        SPLIT_FILE.unlink()
        splits = None
else:
    splits = None

if splits is None:
    train_val, test_rows = train_test_split(
        full_df, test_size=0.15, stratify=full_df.label, random_state=SEED
    )
    train_rows, val_rows = train_test_split(
        train_val, test_size=0.15 / 0.85, stratify=train_val.label, random_state=SEED
    )
    splits = pd.concat([
        train_rows.assign(split="train"), val_rows.assign(split="val"), test_rows.assign(split="test")
    ], ignore_index=True)
    splits.to_csv(SPLIT_FILE, index=False)

display(pd.crosstab(splits.split, splits.label).iloc[:, :8])
print(splits.split.value_counts())

NO_HAND_CLASS = canonical_label("nothing", class_names)
if NO_HAND_CLASS is None:
    raise ValueError("El dataset debe incluir la clase `nothing` para manejar imágenes sin mano.")
class_to_id = {name: i for i, name in enumerate(class_names)}

train_all = splits[splits.split == "train"].copy()
val_all = splits[splits.split == "val"].copy()
test_rows = splits[splits.split == "test"].copy()
train_rows = train_all[(train_all.detected.astype(bool)) | (train_all.label.str.lower() == NO_HAND_CLASS.lower())].copy()
val_rows = val_all[(val_all.detected.astype(bool)) | (val_all.label.str.lower() == NO_HAND_CLASS.lower())].copy()
print(f"Entrenamiento para CNN: {len(train_rows):,}/{len(train_all):,}; validación: {len(val_rows):,}/{len(val_all):,}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def augment_map(image):
    """Brillo/contraste solo en el esqueleto; conserva el fondo negro. Sin rotación ni flips."""
    foreground = tf.reduce_max(image, axis=-1, keepdims=True) > 0.5
    image = tf.image.random_contrast(image, lower=0.80, upper=1.20)
    image = tf.image.random_brightness(image, max_delta=18.0)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return tf.where(foreground, image, tf.zeros_like(image))


def make_dataset(frame, training=False):
    map_paths = [str(FULL_MAPS / p) for p in frame.map_path.astype(str)]
    labels = frame.label.map(class_to_id).to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((map_paths, labels))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    def load_png(path, label):
        image = tf.io.decode_png(tf.io.read_file(path), channels=3)
        image = tf.image.resize(image, (MAP_SIZE, MAP_SIZE), antialias=True)
        image = tf.cast(image, tf.float32)
        return image, label

    ds = ds.map(load_png, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda image, label: (augment_map(image), label), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_rows, training=True)
val_ds = make_dataset(val_rows)
test_ds = make_dataset(test_rows)
print("Mapas por lote:", next(iter(train_ds))[0].shape)

## 6. Modelos y entrenamiento

Ambas redes reciben el mismo mapa RGB de 224 × 224. Se reemplaza su clasificador ImageNet por una cabeza para las clases ASL. Primero se entrena la cabeza con la base congelada; luego se ajustan las últimas capas, manteniendo Batch Normalization congelada. Se guardan curvas, modelo y métricas en Drive.

In [ ]:
if tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")


def build_transfer_model(model_name):
    input_tensor = tf.keras.Input(shape=(MAP_SIZE, MAP_SIZE, 3), name="landmark_map")
    if model_name == "mobilenetv2":
        preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(input_tensor)
        backbone = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(MAP_SIZE, MAP_SIZE, 3)
        )
    elif model_name == "efficientnetb0":
        # EfficientNet de Keras incluye su reescalado interno; recibe píxeles en 0..255.
        preprocessed = input_tensor
        backbone = tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=(MAP_SIZE, MAP_SIZE, 3)
        )
    else:
        raise ValueError(f"Modelo no soportado: {model_name}")

    backbone.trainable = False
    features = backbone(preprocessed, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)
    features = tf.keras.layers.Dropout(0.30)(features)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax", dtype="float32")(features)
    model = tf.keras.Model(input_tensor, outputs, name=model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, backbone


def val_accuracy(model):
    probs = model.predict(val_ds, verbose=0)
    y_true = val_rows.label.map(class_to_id).to_numpy()
    return accuracy_score(y_true, probs.argmax(axis=1))


def evaluate_pipeline(model, frame, output_stem):
    ds = make_dataset(frame)
    probabilities = model.predict(ds, verbose=0)
    y_true = frame.label.map(class_to_id).to_numpy()
    detected = frame.detected.astype(bool).to_numpy()
    y_pred = probabilities.argmax(axis=1)
    no_hand_id = class_to_id[NO_HAND_CLASS]
    y_pred[~detected] = no_hand_id
    sign_mask = frame.label.str.lower().to_numpy() != NO_HAND_CLASS.lower()
    metrics = {
        "accuracy_end_to_end": float(accuracy_score(y_true, y_pred)),
        "macro_f1_end_to_end": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "hand_detection_rate_on_signs": float(detected[sign_mask].mean()),
        "missed_sign_hands": int((~detected[sign_mask]).sum()),
        "n_images": int(len(frame)),
        "classification_report": classification_report(
            y_true, y_pred, labels=list(range(len(class_names))), target_names=class_names,
            output_dict=True, zero_division=0,
        ),
    }
    results = frame[["path", "label", "detected"]].copy()
    results["prediction"] = [class_names[i] for i in y_pred]
    results["correct"] = y_true == y_pred
    results.to_csv(output_stem.with_name(output_stem.name + "_predictions.csv"), index=False)
    with open(output_stem.with_name(output_stem.name + "_metrics.json"), "w", encoding="utf-8") as f:
        import json
        json.dump(metrics, f, indent=2)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))), normalize="true")
    fig, ax = plt.subplots(figsize=(11, 10))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", xticks_rotation=90, values_format=".2f", colorbar=False
    )
    ax.set_title("Flujo completo: mapa MediaPipe + CNN")
    fig.tight_layout()
    fig.savefig(output_stem.with_name(output_stem.name + "_confusion.png"), dpi=140)
    plt.show()
    return metrics


def train_one(model_name):
    run_dir = OUTPUT_DIR / "experiments" / model_name
    run_dir.mkdir(parents=True, exist_ok=True)
    model, backbone = build_transfer_model(model_name)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
    ]
    head_history = model.fit(
        train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=callbacks, verbose=2
    )
    head_score = val_accuracy(model)
    head_path = run_dir / "head_model.keras"
    model.save(head_path)

    backbone.trainable = True
    unfreeze_last = 30 if model_name == "mobilenetv2" else 40
    for layer in backbone.layers[:-unfreeze_last]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    fine_history = model.fit(
        train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS,
        callbacks=callbacks, verbose=2,
    )
    fine_score = val_accuracy(model)
    if fine_score < head_score:
        print(f"Fine-tuning no mejoró validación ({fine_score:.4f} < {head_score:.4f}); se conserva la cabeza congelada.")
        model = tf.keras.models.load_model(head_path)
    else:
        print(f"Se conserva fine-tuning (val_accuracy={fine_score:.4f}).")

    model_path = run_dir / "model.keras"
    model.save(model_path)
    history = pd.concat([
        pd.DataFrame(head_history.history).assign(fase="cabeza"),
        pd.DataFrame(fine_history.history).assign(fase="fine_tuning"),
    ], ignore_index=True)
    history.insert(0, "epoch", np.arange(1, len(history) + 1))
    history.to_csv(run_dir / "history.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Pérdida"]):
        ax.plot(history.epoch, history[metric], label="train")
        ax.plot(history.epoch, history[f"val_{metric}"], label="validación")
        ax.axvline(len(head_history.history["loss"]) + 0.5, color="gray", linestyle=":", label="inicio fine-tuning")
        ax.set_title(title); ax.set_xlabel("Época"); ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); fig.savefig(run_dir / "curves.png", dpi=140); plt.show()

    metrics = evaluate_pipeline(model, test_rows, run_dir / "internal_test")
    metrics.update({"model": model_name, "params": int(model.count_params()), "head_val_accuracy": float(head_score), "fine_tune_val_accuracy": float(fine_score)})
    with open(run_dir / "summary.json", "w", encoding="utf-8") as f:
        import json
        json.dump(metrics, f, indent=2)
    print(
        f"{model_name}: accuracy end-to-end={metrics['accuracy_end_to_end']:.4f} | "
        f"macro F1={metrics['macro_f1_end_to_end']:.4f} | "
        f"detección de manos en señas={metrics['hand_detection_rate_on_signs']:.1%}"
    )
    return model, metrics, run_dir

## 7. Entrenar y comparar MobileNetV2 con EfficientNetB0

La celda siguiente ejecuta ambos modelos secuencialmente y guarda cada corrida en su carpeta. Puede tardar varias horas según el tamaño real del dataset, la velocidad de extracción y la GPU de Colab.

In [ ]:
all_metrics = []
for model_name in ["mobilenetv2", "efficientnetb0"]:
    trained_model, metrics, run_dir = train_one(model_name)
    all_metrics.append(metrics)
    print("Resultados:", run_dir)
    del trained_model
    tf.keras.backend.clear_session()

summary = pd.DataFrame([
    {
        "modelo": m["model"],
        "accuracy end-to-end": m["accuracy_end_to_end"],
        "macro F1 end-to-end": m["macro_f1_end_to_end"],
        "detección de mano en señas": m["hand_detection_rate_on_signs"],
        "manos de señas no detectadas": m["missed_sign_hands"],
        "parámetros (M)": round(m["params"] / 1e6, 2),
    }
    for m in all_metrics
])
summary.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
display(summary)

## 8. Evaluación opcional con fotos propias

Si `OWN_PHOTOS_DIR` existe y tiene una subcarpeta por clase, se generan mapas externos (también con caché) y se evalúa cada modelo. Es un conjunto externo: no lo uses para ajustar hiperparámetros repetidamente y luego reportarlo como prueba final.

In [ ]:
if OWN_PHOTOS_DIR.is_dir():
    OWN_MAPS = OUTPUT_DIR / "maps_own_photos"
    OWN_MANIFEST = OUTPUT_DIR / "maps_own_photos_manifest.csv"
    own_df = prepare_maps(
        OWN_PHOTOS_DIR, OWN_MAPS, OWN_MANIFEST, MODEL_ASSET,
        names=class_names, max_per_class=None, progress_every=100,
    ).drop_duplicates("path", keep="last").reset_index(drop=True)
    print(f"Fotos propias: {len(own_df)} | detección global: {own_df.detected.mean():.1%}")
    own_results = []
    for model_name in ["mobilenetv2", "efficientnetb0"]:
        model_path = OUTPUT_DIR / "experiments" / model_name / "model.keras"
        model = tf.keras.models.load_model(model_path)
        # Evaluator usa FULL_MAPS; cambia temporalmente la raíz al conjunto externo.
        previous_maps = FULL_MAPS
        FULL_MAPS = OWN_MAPS
        own_metrics = evaluate_pipeline(model, own_df, OUTPUT_DIR / "experiments" / model_name / "own_photos")
        FULL_MAPS = previous_maps
        own_results.append({
            "modelo": model_name,
            "accuracy end-to-end fotos propias": own_metrics["accuracy_end_to_end"],
            "macro F1 fotos propias": own_metrics["macro_f1_end_to_end"],
            "detección en señas": own_metrics["hand_detection_rate_on_signs"],
        })
        del model
    own_summary = pd.DataFrame(own_results)
    own_summary.to_csv(OUTPUT_DIR / "own_photos_comparison.csv", index=False)
    display(own_summary)
else:
    print("No hay carpeta de fotos propias en OWN_PHOTOS_DIR; se omite esta sección.")

## Interpretación y límites

- Compara primero **macro F1** y accuracy end-to-end: el segundo cuenta como error las señas para las que MediaPipe no encontró mano. Revisa también la cobertura del detector y las matrices por modelo.
- La matriz de puntos elimina texturas y gran parte del fondo; por eso este experimento estudia la forma de la mano. También elimina color y detalles de la imagen, así que una caída de rendimiento puede indicar que esos detalles aportaban información (por ejemplo, orientación o autooclusión).
- Las augmentaciones de brillo/contraste se aplican al dibujo y mantienen el fondo negro; no se rota ni se voltea porque podría cambiar la seña.
- MediaPipe no usa GPU necesariamente en este flujo de Python. La inferencia de la CNN sí puede aprovechar la GPU.
- Las señas dinámicas, como letras cuyo significado depende del movimiento, requieren secuencias de video y quedan fuera de este experimento estático.